In [17]:
import numpy as np
import matplotlib.pyplot as plt

import essentia.standard as es
import librosa

In [18]:
from IPython.display import Audio
from pathlib import Path
from pprint import pprint

In [19]:
from perturbations.data.audio_io import load_performances, load_audio
from perturbations.consts import SAMPLE_RATE

In [20]:
# loading performances

audio_dir = Path("../audio/ween_rosesarefree/raw")
performances = load_performances(audio_dir)

sr = SAMPLE_RATE

Loaded 9 files


In [21]:
# percussive separation function 
def get_percussive_source(audio):
  D = librosa.stft(audio)
  _, D_percussive = librosa.decompose.hpss(D)
  y_percussive = librosa.istft(D_percussive, length=len(audio))
  return y_percussive

In [22]:
# beat extraction

rhythm_extractor = es.RhythmExtractor2013(method="multifeature") # type: ignore

bpm_results_essentia = {}
confidence_results_essentia = {}
for name, perf in performances.items():
  bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(perf)
  bpm_results_essentia[name] = bpm
  confidence_results_essentia[name] = beats_confidence
  print("BPM:", bpm)
  print("Beat estimation confidence:", beats_confidence)



BPM: 184.57081604003906
Beat estimation confidence: 2.0662448406219482
BPM: 184.5707244873047
Beat estimation confidence: 2.1101226806640625
BPM: 178.20550537109375
Beat estimation confidence: 1.911846399307251
BPM: 178.20529174804688
Beat estimation confidence: 1.4448407888412476
BPM: 89.8626937866211
Beat estimation confidence: 1.9238917827606201
BPM: 90.59420776367188
Beat estimation confidence: 1.996793508529663
BPM: 184.5707244873047
Beat estimation confidence: 0.5014057755470276
BPM: 90.7189712524414
Beat estimation confidence: 1.6948715448379517
BPM: 89.94955444335938
Beat estimation confidence: 1.5407733917236328


In [23]:
# beat extraction with source separation

rhythm_extractor = es.RhythmExtractor2013(method="multifeature") # type: ignore

bpm_results_essentia_perc = {}
confidence_results_essentia_perc = {}
for name, perf in performances.items():
  perf_perc = get_percussive_source(perf)
  bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(perf_perc)
  bpm_results_essentia_perc[name] = bpm
  confidence_results_essentia_perc[name] = beats_confidence
  print("BPM:", bpm)
  print("Beat estimation confidence:", beats_confidence)



BPM: 184.57064819335938
Beat estimation confidence: 2.331477403640747
BPM: 184.57081604003906
Beat estimation confidence: 2.227386474609375
BPM: 178.20545959472656
Beat estimation confidence: 2.242269992828369
BPM: 90.01371765136719
Beat estimation confidence: 1.9009593725204468
BPM: 90.00765228271484
Beat estimation confidence: 2.9566445350646973
BPM: 184.57073974609375
Beat estimation confidence: 1.9182937145233154
BPM: 90.79545593261719
Beat estimation confidence: 1.4531676769256592
BPM: 90.7231216430664
Beat estimation confidence: 1.984592080116272
BPM: 90.09590148925781
Beat estimation confidence: 1.8528318405151367


In [26]:
bpm_results_essentia_cleaned = {k: v/2 if v > 100 else v for k, v in bpm_results_essentia.items()}
bpm_results_essentia_perc_cleaned = {k: v/2 if v > 100 else v for k, v in bpm_results_essentia_perc.items()}

pprint(bpm_results_essentia_cleaned)
pprint(bpm_results_essentia_perc_cleaned)

pprint(sum(confidence_results_essentia.values()))
pprint(sum(confidence_results_essentia_perc.values()))

{'1-31-2008 Part_0108': 89.10264587402344,
 'ween00.05.18d1T14': 92.28536224365234,
 'ween1999-08-02t17': 90.7189712524414,
 'ween2000-05-08-d1t10': 92.28540802001953,
 'ween2000-05-09d1t05': 92.28536224365234,
 'ween2000-06-18d1t13': 90.59420776367188,
 'ween2003-10-03d1t17': 89.10275268554688,
 'ween2004-10-16_wiltern_t22': 89.94955444335938,
 'ween2008-03-07t17': 89.8626937866211}
{'1-31-2008 Part_0108': 90.01371765136719,
 'ween00.05.18d1T14': 92.28540802001953,
 'ween1999-08-02t17': 90.7231216430664,
 'ween2000-05-08-d1t10': 92.28532409667969,
 'ween2000-05-09d1t05': 90.79545593261719,
 'ween2000-06-18d1t13': 92.28536987304688,
 'ween2003-10-03d1t17': 89.10272979736328,
 'ween2004-10-16_wiltern_t22': 90.09590148925781,
 'ween2008-03-07t17': 90.00765228271484}
15.190790712833405
18.86762309074402


In [25]:
# get bpm with essentia

rhythm_extractor = es.RhythmExtractor2013(method="multifeature")
bpm, beats, beats_confidence, _, beats_intervals = rhythm_extractor(audio)

bpm_results_essentia = {}
for name, perf in performances.items():
    tempo, beats = librosa.beat.beat_track(y=perf, sr=sr)
    bpm_results_librosa[name] = tempo

NameError: name 'audio' is not defined

In [ ]:
# show librosa bpm results
pprint(bpm_results_librosa)

{'1-31-2008 Part_0108': array([89.10290948]),
 'ween00.05.18d1T14': array([92.28515625]),
 'ween1999-08-02t17': array([92.28515625]),
 'ween2000-05-08-d1t10': array([92.28515625]),
 'ween2000-05-09d1t05': array([92.28515625]),
 'ween2000-06-18d1t13': array([92.28515625]),
 'ween2003-10-03d1t17': array([89.10290948]),
 'ween2004-10-16_wiltern_t22': array([89.10290948]),
 'ween2008-03-07t17': array([89.10290948])}
